# Notebook 022: Advanced Aggregations: collect_list vs collect_set and Sorted Collections

**Module:** Module 02: Complex Data Types & Nested Structures Handling  
**Difficulty:** `Intermediate`  
**Databricks Masterclass Reference:** Video 06: Master Databricks Notebooks & Video 23: Interview Questions  

---

## 1. Business Scenario & Real-World Context
A digital customer journey pipeline aggregates all URLs visited by users within a session. Product teams need both: (1) The exact chronologically ordered sequence of pages visited (including duplicates), and (2) The unique set of pages visited without duplicates.

## 2. Technical Requirement & Problem Statement
Compare `collect_list()` (preserves duplicates) vs `collect_set()` (drops duplicates). Demonstrate sorting collected arrays chronologically by sorting before aggregation or using `array_sort` on collected structs.

## 3. Spark & Databricks Architecture Notes
`collect_list` and `collect_set` are wide aggregations that buffer array elements in executor memory per group. If a single key contains millions of entries (key skew), this can trigger severe Executor OutOfMemory (OOM) errors. Pre-sorting rows before `groupBy` is NOT guaranteed to preserve order across shuffles unless using sorted structs.


## 4. Environment Setup & SparkSession Initialization
We configure a resilient `SparkSession` compatible with local PySpark development, Databricks runtime, or CI/CD test runners.

In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# Initialize SparkSession with Delta Lake support and optimized local configs
spark = (
    SparkSession.builder
    .appName(f"UseCase_022_Advanced_Aggregations_collect_list_vs_collect_set_and_Sorted_Collections")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.default.parallelism", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f" Spark Version: {spark.version}")
print(f" Session Active: {spark.sparkContext.appName}")


## 5. Generating User Session Clickstream
Create chronologically timestamped page views per user session.

In [ ]:
clickstream = [
    ("SESS-1", "2023-11-20 10:00:00", "/home"),
    ("SESS-1", "2023-11-20 10:02:00", "/products"),
    ("SESS-1", "2023-11-20 10:05:00", "/products"),  # Repeated visit
    ("SESS-1", "2023-11-20 10:10:00", "/checkout"),
    ("SESS-2", "2023-11-20 11:00:00", "/home"),
    ("SESS-2", "2023-11-20 11:04:00", "/pricing"),
    ("SESS-2", "2023-11-20 11:08:00", "/home")       # Repeated visit
]

schema = "session_id STRING, event_time STRING, page_url STRING"
df_clicks = spark.createDataFrame(clickstream, schema)
df_clicks.show()

## 6. collect_list vs collect_set Aggregation
Observe difference in duplicate retention.

In [ ]:
df_aggregated = df_clicks.groupBy("session_id").agg(
    F.collect_list("page_url").alias("all_pages_list"),
    F.collect_set("page_url").alias("unique_pages_set")
)

df_aggregated.show(truncate=False)

## 7. Guaranteed Chronological Sorting of Collected Elements
Pack timestamp and URL into a struct, collect, sort, and extract page URLs.

In [ ]:
# Pack timestamp and URL into struct: array_sort will sort by struct's first field (event_time)
df_sorted_journey = (
    df_clicks
    .groupBy("session_id")
    .agg(
        F.array_sort(F.collect_list(F.struct("event_time", "page_url"))).alias("sorted_struct_events")
    )
    .withColumn(
        "ordered_page_sequence",
        F.expr("transform(sorted_struct_events, x -> x.page_url)")
    )
    .select("session_id", "ordered_page_sequence")
)

print("Guaranteed Chronologically Ordered Customer Journey:")
df_sorted_journey.show(truncate=False)

## 9. Key Architectural Takeaways
- **`collect_list()` retains duplicates and element count; `collect_set()` removes duplicates without guaranteed order.**
- **Shuffles in Spark do not guarantee row arrival order; to sort collected arrays, bundle values into structs with sort keys and apply `array_sort()`.**
- **Beware of key skew: collecting millions of items on a single key causes executor heap exhaustion.**

## 10. Senior Data Engineering Interview Questions
1. Why is `df.sort('time').groupBy('id').agg(collect_list('val'))` NOT guaranteed to preserve order in distributed Spark?
2. How do you safely sort elements collected by `collect_list()`?
3. What risk does `collect_list()` pose when operating on skewed keys?

---
*Created for the **Spark Coding Use Cases Masterclass** repository referencing the Databricks All-in-One Playlist.*
